In [1]:
# === Setup: paths + helper ===
from pathlib import Path
import time, pandas as pd
from project_package.modeling import (
    train_classification_from_csv,
    train_regression_from_csv,
)

ROOT = Path.cwd()
OUT  = ROOT / "supervised"
OUT.mkdir(parents=True, exist_ok=True)

CSV = ROOT / "ncr_ride_bookings_with_weather_filled_scaled_short.csv"
if not CSV.exists():
    alt = ROOT / "datasets" / CSV.name
    if alt.exists(): CSV = alt
assert CSV.exists(), f"CSV not found: {CSV}"

IDS = ["Booking ID", "Customer ID"]  # keep in CSV exports

def timed(fn, **kw):
    t0 = time.time()
    res = fn(**kw)
    mins = (time.time() - t0) / 60
    return res, mins

def save_report(report: dict, path: Path):
    pd.DataFrame([report]).to_csv(path, index=False)


In [2]:
# === Problem #1: Classification (completion vs cancellation) ===
cls_res, t_cls = timed(
    train_classification_from_csv,
    csv_path=str(CSV),
    target_col=None,            # derive _target_completed_ from "Booking Status"
    id_cols=IDS,
    artifacts_dir=str(OUT),
    valid_ratio=0.20,
    random_state=42,
    tune_row_cap=40000,
)
save_report(cls_res.report, OUT / f"cls_best_{cls_res.best_model_name}_report.csv")

print("=== Completion (classification) ===")
print("time (min):", round(t_cls, 2))
print("best model:", cls_res.best_model_name)
print("report   :", cls_res.report)
print("preds CSV:", cls_res.preds_csv_path)
print("split CSV:", cls_res.split_csv_path)
print("model PKL:", cls_res.model_path)


=== Completion (classification) ===
time (min): 34.59
best model: rf
report   : {'model': 'rf', 'accuracy': 0.9374893428366811, 'precision': 0.9126375290198849, 'recall': 0.9943909815782238, 'f1': 0.9517618884707494, 'roc_auc': 0.9684372369868492}
preds CSV: c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\supervised\cls_best_rf_preds.csv
split CSV: c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\supervised\split_assignments_classification.csv
model PKL: c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\supervised\best_cls_rf__target_completed_.pkl


In [3]:
# === Problem #2: Regression (fare) ===
fare_target = "Booking Value_fill_scaled"   # switch to unscaled if your team prefers
fare_res, t_fare = timed(
    train_regression_from_csv,
    csv_path=str(CSV),
    target_col=fare_target,
    id_cols=IDS,
    artifacts_dir=str(OUT),
    valid_ratio=0.20,
    random_state=42,
    tune_row_cap=40000,
)
save_report(fare_res.report, OUT / f"reg_best_{fare_target.replace(' ','_')}_{fare_res.best_model_name}_report.csv")

print("=== Fare (regression) ===")
print("time (min):", round(t_fare, 2))
print("best model:", fare_res.best_model_name)
print("report   :", fare_res.report)
print("preds CSV:", fare_res.preds_csv_path)
print("split CSV:", fare_res.split_csv_path)
print("model PKL:", fare_res.model_path)


=== Fare (regression) ===
time (min): 27.22
best model: tree
report   : {'model': 'tree', 'MAE': 0.986091067841411, 'RMSE': 1.6794663952263587, 'R2': -0.0005764767983282848}
preds CSV: c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\supervised\reg_best_Booking_Value_fill_scaled_tree_preds.csv
split CSV: c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\supervised\split_assignments_regression_Booking_Value_fill_scaled.csv
model PKL: c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\supervised\best_reg_tree_Booking Value_fill_scaled.pkl


In [4]:
# === Problem #3: Regression (customer rating) ===
rating_target = "Customer Rating_fill"
rating_res, t_rating = timed(
    train_regression_from_csv,
    csv_path=str(CSV),
    target_col=rating_target,
    id_cols=IDS,
    artifacts_dir=str(OUT),
    valid_ratio=0.20,
    random_state=42,
    tune_row_cap=40000,
)
save_report(rating_res.report, OUT / f"reg_best_{rating_target.replace(' ','_')}_{rating_res.best_model_name}_report.csv")

print("=== Rating (regression) ===")
print("time (min):", round(t_rating, 2))
print("best model:", rating_res.best_model_name)
print("report   :", rating_res.report)
print("preds CSV:", rating_res.preds_csv_path)
print("split CSV:", rating_res.split_csv_path)
print("model PKL:", rating_res.model_path)


=== Rating (regression) ===
time (min): 26.89
best model: tree
report   : {'model': 'tree', 'MAE': 0.216340046624083, 'RMSE': 0.3469420037365729, 'R2': -0.0005331047369210307}
preds CSV: c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\supervised\reg_best_Customer_Rating_fill_tree_preds.csv
split CSV: c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\supervised\split_assignments_regression_Customer_Rating_fill.csv
model PKL: c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\supervised\best_reg_tree_Customer Rating_fill.pkl


The below is the lite version.  Choose the 15-25 best features to model and check the performance.
Also, not using all the records((15-30%) of total data)) to reduce computation time.

In [1]:
# === LITE setup: robust helpers, paths, and knobs ===
from pathlib import Path
import os, time, pandas as pd, numpy as np

# Import modeling utilities from your project package
from project_package.modeling import (
    EXCLUDE_ALWAYS, load_csv_dedup, make_binary_target,
    train_classification_from_csv, train_regression_from_csv
)

# Resolve data path (repo root first, then ./datasets/)
ROOT = Path.cwd()
CSV = ROOT / "ncr_ride_bookings_with_weather_filled_scaled_short.csv"
if not CSV.exists():
    alt = ROOT / "datasets" / CSV.name
    if alt.exists():
        CSV = alt
assert CSV.exists(), f"CSV not found: {CSV}"

# Output folder for this LITE run (kept separate from full runs)
OUT = ROOT / "supervised_lite"
OUT.mkdir(parents=True, exist_ok=True)

# ID columns to carry through in exported CSVs (not used as features)
IDS = ["Booking ID", "Customer ID"]

# LITE knobs: reduce features and rows to speed up retraining/evaluations
K_FEATURES   = 20      # keep top-K numeric features by |corr with target| (suggest 15–25)
SAMPLE_FRAC  = 0.30    # use 15–30% of rows; stratified if target is binary
RANDOM_STATE = 42      # reproducibility

def topk_numeric_corr_view(csv_path: Path, target_col: str | None, k: int, sample_frac: float) -> Path:
    """
    Build a reduced CSV for fast experimentation:
      1) If target_col is None, derive binary target `_target_completed_` from Booking Status.
      2) Rank numeric features by absolute correlation with the target; keep top-K.
      3) Keep [IDS + target + top-K numeric features].
      4) Row sampling (stratified if target is binary 0/1).
    Returns the path to the reduced CSV next to the original.
    """
    df = load_csv_dedup(str(csv_path))

    # Ensure we have a target column
    if (target_col is None) or (target_col not in df.columns):
        df, target_col = make_binary_target(df)  # creates `_target_completed_`

    # Features for correlation scoring: exclude leakage, IDs, and the target itself
    drop_cols = set(EXCLUDE_ALWAYS) | set(IDS) | {target_col}
    X = df.drop(columns=[c for c in drop_cols if c in df.columns], errors="ignore")

    # Rank numeric features by |corr| with the target
    num_cols = X.select_dtypes(include=["number"]).columns.tolist()
    corr = df[num_cols + [target_col]].corr(numeric_only=True)[target_col].abs().dropna().sort_values(ascending=False)
    keep_num = corr.head(k).index.tolist()

    # Build reduced view that includes IDs + target + top-K numeric features
    cols = IDS + [target_col] + keep_num
    df_small = df[[c for c in cols if c in df.columns]].copy()

    # Row sampling (stratified if target is binary)
    if sample_frac < 1.0:
        y = df_small[target_col]
        if isinstance(y, pd.DataFrame):  # force Series if a DataFrame slipped through
            y = y.iloc[:, 0]
        y_num = pd.to_numeric(y, errors="coerce")
        uniq = pd.unique(y_num.dropna())
        if set(uniq).issubset({0, 1}):   # stratify by binary target
            df_small = (
                df_small
                .groupby(y_num, group_keys=False)
                .apply(lambda g: g.sample(frac=sample_frac, random_state=RANDOM_STATE))
                .reset_index(drop=True)
            )
        else:
            df_small = df_small.sample(frac=sample_frac, random_state=RANDOM_STATE).reset_index(drop=True)

    out = csv_path.with_name(csv_path.stem + f"__top{k}_s{int(sample_frac*100)}.csv")
    df_small.to_csv(out, index=False)
    return out

def run_and_print(train_fn, **kw):
    """
    Wrapper to train and print key runtime info:
      - training minutes
      - model file size (MB)
      - artifact paths (preds CSV, split CSV, model PKL)
    """
    t0 = time.time()
    res = train_fn(**kw)
    mins = (time.time() - t0) / 60
    size_mb = os.path.getsize(res.model_path) / (1024 * 1024)
    print(f"  time (min): {mins:.2f}")
    print(f"  model size: {size_mb:.1f} MB")
    print(f"  preds CSV : {res.preds_csv_path}")
    print(f"  split CSV : {res.split_csv_path}")
    print(f"  model PKL : {res.model_path}")
    return res


In [2]:
# === LITE: Problem #1 — Classification (completion vs cancellation) ===
# Build a reduced CSV specifically for the classification target
csv_lite_cls = topk_numeric_corr_view(
    CSV, target_col=None, k=K_FEATURES, sample_frac=SAMPLE_FRAC
)
print("=== LITE: Completion (classification) ===")

# IMPORTANT: target is already derived as `_target_completed_` in the reduced CSV
_ = run_and_print(
    train_classification_from_csv,
    csv_path=str(csv_lite_cls),
    target_col="_target_completed_",   # explicitly specify the derived target
    id_cols=IDS,
    artifacts_dir=str(OUT),
    valid_ratio=0.20,
    random_state=RANDOM_STATE,
    tune_row_cap=None,                 # no row cap here because the data is already lite
)


=== LITE: Completion (classification) ===
  time (min): 0.28
  model size: 0.0 MB
  preds CSV : c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\supervised_lite\cls_best_logreg_preds.csv
  split CSV : c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\supervised_lite\split_assignments_classification.csv
  model PKL : c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\supervised_lite\best_cls_logreg__target_completed_.pkl


In [3]:
# === LITE: Problem #2 — Regression (fare) ===
fare_tgt = "Booking Value_fill_scaled"

# Build a reduced CSV for the fare regression target
csv_lite_fare = topk_numeric_corr_view(
    CSV, target_col=fare_tgt, k=K_FEATURES, sample_frac=SAMPLE_FRAC
)
print("=== LITE: Fare (regression) ===")

_ = run_and_print(
    train_regression_from_csv,
    csv_path=str(csv_lite_fare),
    target_col=fare_tgt,
    id_cols=IDS,
    artifacts_dir=str(OUT),
    valid_ratio=0.20,
    random_state=RANDOM_STATE,
    tune_row_cap=None,
)


=== LITE: Fare (regression) ===
  time (min): 0.16
  model size: 0.0 MB
  preds CSV : c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\supervised_lite\reg_best_Booking_Value_fill_scaled_ridge_preds.csv
  split CSV : c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\supervised_lite\split_assignments_regression_Booking_Value_fill_scaled.csv
  model PKL : c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\supervised_lite\best_reg_ridge_Booking Value_fill_scaled.pkl


In [4]:
# === LITE: Problem #3 — Regression (customer rating) ===
rating_tgt = "Customer Rating_fill"

# Build a reduced CSV for the rating regression target
csv_lite_rating = topk_numeric_corr_view(
    CSV, target_col=rating_tgt, k=K_FEATURES, sample_frac=SAMPLE_FRAC
)
print("=== LITE: Rating (regression) ===")

_ = run_and_print(
    train_regression_from_csv,
    csv_path=str(csv_lite_rating),
    target_col=rating_tgt,
    id_cols=IDS,
    artifacts_dir=str(OUT),
    valid_ratio=0.20,
    random_state=RANDOM_STATE,
    tune_row_cap=None,
)


=== LITE: Rating (regression) ===
  time (min): 0.15
  model size: 0.0 MB
  preds CSV : c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\supervised_lite\reg_best_Customer_Rating_fill_tree_preds.csv
  split CSV : c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\supervised_lite\split_assignments_regression_Customer_Rating_fill.csv
  model PKL : c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\supervised_lite\best_reg_tree_Customer Rating_fill.pkl


=== ARTIFACTS (lite) ===


,dir,file,ext,size_mb,mtime
0,supervised_lite,supervised_lite\cls_best_logreg_preds.csv,.csv,0.14,2025-10-06T21:52:59
1,supervised_lite,supervised_lite\cls_best_logreg_report.csv,.csv,0.00,2025-10-06T21:52:59
2,supervised_lite,supervised_lite\reg_best_Booking_Value_fill_sc...,.csv,0.60,2025-10-06T21:53:12
3,supervised_lite,supervised_lite\reg_best_Customer_Rating_fill_...,.csv,0.52,2025-10-06T21:53:25
4,supervised_lite,supervised_lite\split_assignments_classificati...,.csv,0.46,2025-10-06T21:52:59
5,supervised_lite,supervised_lite\split_assignments_regression_B...,.csv,2.50,2025-10-06T21:53:12
6,supervised_lite,supervised_lite\split_assignments_regression_C...,.csv,2.43,2025-10-06T21:53:25
7,supervised_lite,supervised_lite\best_cls_logreg__target_comple...,.pkl,0.00,2025-10-06T21:52:59
8,supervised_lite,supervised_lite\best_reg_ridge_Booking Value_f...,.pkl,0.00,2025-10-06T21:53:12
9,supervised_lite,supervised_lite\best_reg_tree_Customer Rating_...,.pkl,0.01,2025-10-06T21:53:25



[Classification: completion] samples:


,file,size_mb
0,supervised_lite\cls_best_logreg_preds.csv,0.14
1,supervised_lite\cls_best_logreg_report.csv,0.00
7,supervised_lite\best_cls_logreg__target_comple...,0.00


[Regression: fare] samples:


,file,size_mb
2,supervised_lite\reg_best_Booking_Value_fill_sc...,0.6


[Regression: rating] samples:


,file,size_mb
3,supervised_lite\reg_best_Customer_Rating_fill_...,0.52



Saved audit CSV -> c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\supervised_lite\audit_artifacts_lite.csv
